# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**Research question:** Among a client's already-published, indexed pages, which ones are likely to see a
*meaningful drop in organic search demand* over the next two weeks, early enough that a content team can
intervene (refresh, re-optimize, or re-promote) before the drop shows up in the client's monthly report?

**Decision this supports:** which pages to put at the top of a content team's weekly refresh queue.

**Action taken on the decision:** an editor opens the top-ranked pages, checks title/snippet/on-page content
against current search intent, and either refreshes the page or explicitly decides it isn't worth the effort.

**Cost of a wrong call:**
- *False positive* (flagged as declining, isn't really): a few hours of an editor's time spent reviewing a
  page that didn't need it. Annoying, not dangerous.
- *False negative* (a real decliner not flagged): a page keeps losing visibility for another reporting cycle
  before anyone notices — a slower, quieter cost, but the queue is a ranked list, not a filter, so a missed
  page is one that simply didn't reach the top rather than one that's invisible forever.

Because false positives are cheap and false negatives are only "delayed," this is framed as a **ranking /
scoring problem** (Precision@K on a prioritized queue), not a strict binary classifier a team must obey.


In [1]:
import duckdb, pandas as pd, numpy as np, os

# Portable path: works both in the sandboxed execution environment and if this
# notebook is reopened locally on the machine that downloaded the release.
candidates = [
    os.path.expanduser("~/Documents/flyrank-hf-data"),
    os.path.expanduser("~/mnt/Documents/flyrank-hf-data"),
]
BASE = next((p for p in candidates if os.path.isdir(p)), candidates[0])

con = duckdb.connect()
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

# Release: March 2026 warehouse partition (the same partition used in ML-04/ML-05).
# Window split in half: days 1-15 as the "prior" observation window, days 16-31 (16 days) as the
# "target" window whose demand we are trying to predict will drop.
# Restricted to rows where gsc_data_available IS TRUE (GSC coverage is ~37% of rows in this
# partition per the ML-04 data contract - the rest is excluded rather than imputed).
print(f"Reading local parquet from: {BASE}")

# Repo root: robust whether this notebook is executed by copying it to the repo root (this run)
# or reopened later at its normal location work/notebooks/capstone.ipynb.
REPO_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "work")) \
    else os.path.abspath(os.path.join(os.getcwd(), "..", ".."))


Reading local parquet from: /sessions/rcw-01sg1afdfkqjrxmdmesxl4zj/mnt/Documents/flyrank-hf-data


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** the March 2026 monthly warehouse partition (`fact_content_daily_performance`,
`month=2026-03`), joined to `dim_content` for content metadata. Same partition used throughout the
weekly assignments, so the data contract already established in ML-04 applies here.

**Date windows:**
- **Prior window** — 2026-03-01 through 2026-03-15 (15 days): where every feature is computed from.
- **Target window** — 2026-03-16 through 2026-03-31 (16 days): where the label is computed from.

Splitting a single month in half in time (rather than a random row split) is what makes this a genuine
"predict the future from the past" setup instead of a same-window correlation exercise.

**What was excluded and why:**
- Rows where `gsc_data_available` is not `TRUE` — about 63% of the partition per the ML-04 contract.
  GSC is the only source with impressions/clicks/position, so a row without it has nothing to compute
  a demand trend from. It is dropped, not imputed, so the queue is silent on those pages rather than
  guessing.
- Pages with fewer than 10 impressions in the prior window — too little signal to say anything about a
  trend; a page with 2 impressions going to 0 is not a "decline," it's noise.
- Pages whose `content_age_days` (from `content_created_date`) computed negative — a data-quality edge
  case (a handful of rows), dropped rather than silently clamped.

No client names, raw URLs, or content text are used or shown anywhere below — only hashed IDs, counts,
rates, and dates.


In [2]:
feat_query = f"""
    WITH avail AS (SELECT * FROM {FACT} WHERE gsc_data_available IS TRUE),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior,
               SUM(gsc_clicks)      AS clicks_prior,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS avg_position_prior,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impr_prior
        FROM avail WHERE report_date <= DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    target AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_target,
               SUM(gsc_clicks) AS clicks_target
        FROM avail WHERE report_date >= DATE '2026-03-16'
        GROUP BY 1, 2
    )
    SELECT p.*, COALESCE(t.impressions_target, 0) AS impressions_target,
           COALESCE(t.clicks_target, 0) AS clicks_target
    FROM prior p LEFT JOIN target t USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior >= 10
"""
feat = con.sql(feat_query).df()

content = con.sql(f"SELECT content_hash_id, word_count, content_created_date, content_type FROM {DIM_CONTENT}").df()
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat["content_created_date"])).dt.days
feat = feat[feat["content_age_days"] >= 0].copy()

# --- Label: is_declining ---
# Daily impression RATE (not raw count) in each window, since the windows are different lengths
# (15 days prior vs 16 days target) - comparing raw sums would bias toward the shorter window.
feat["imp_rate_prior"]  = feat["impressions_prior"]  / 15.0
feat["imp_rate_target"] = feat["impressions_target"] / 16.0
# "Meaningful decline" = at least a 20% drop in daily impression rate. This mirrors the label design
# already used in notebooks/03_working_with_the_full_release.ipynb, so results are comparable across
# the internship's own materials rather than a one-off definition invented just for this capstone.
feat["is_declining"] = (feat["imp_rate_target"] < 0.8 * feat["imp_rate_prior"]).astype(int)
feat["ctr_prior"] = feat["clicks_prior"] / feat["impressions_prior"]

print(f"{len(feat):,} pages / {feat['client_hash_id'].nunique()} clients")
print(f"decline rate (label prevalence): {feat['is_declining'].mean():.3f}")

# NOTE on a leakage trap I caught myself before trusting these numbers: an earlier version of this
# label used CLICKS instead of impressions with a strict "<" comparison. Pages with clicks_prior == 0
# can never see clicks_target go negative, so they were mechanically guaranteed a "not declining" label
# regardless of anything else - a spurious deterministic link between clicks_prior and the label that
# inflated ROC-AUC to ~0.97 while a naive baseline scored *worse than random* (0.14). That mismatch
# (model "too good", baseline "too bad") was the tell. Switching to an impressions-based rate with a
# real 20%-drop threshold (this cell) removed the artifact; see Results below for the honest numbers.


114,715 pages / 39 clients
decline rate (label prevalence): 0.344


## 3. Methodology

**Label — `is_declining`:** 1 if the page's daily impression rate in the target window is at least 20%
lower than its daily impression rate in the prior window, else 0. Rates (not raw counts) are used because
the two windows are different lengths (15 vs 16 days). This is a **current-window proxy** for "losing
organic demand," not a certified long-term decline — a page could recover the following month.

**Features (all computed only from the prior window, so nothing here leaks the future):**
`impressions_prior`, `clicks_prior`, `ctr_prior`, `avg_position_prior`, `days_with_impr_prior`,
`content_age_days`, `content_type` (one-hot encoded).

**Baseline (no model, no fitting on labels):** a CTR-gap rule — the same construction as ML-05. Pages
are bucketed by `avg_position_prior`, an "expected CTR" is computed as the *train-set* average CTR for
that bucket, and `baseline_score = max(0, expected_ctr - ctr_prior) * impressions_prior`. This asks
"how much click volume is this page leaving on the table for where it ranks?", not "will demand drop."

**Model:** `RandomForestClassifier` (300 trees, max_depth=8, `class_weight="balanced"`), trained on the
prior-window features to predict `is_declining`.

**Validation design — client-grouped split, not random:** an 80/20 **`GroupShuffleSplit` grouped by
`client_hash_id`** so every page from a given client lands entirely in train or entirely in test, never
both. A client's pages share editorial style, publishing cadence, and niche, so a random row-level split
would let the model partly memorize "how this client's pages behave" rather than learn a pattern that
generalizes to a client it has never seen — the same memorization-gap concern raised in ML-03/ML-04.
The Results section below reports both the grouped score and a non-grouped comparison to show the size
of that gap directly.

**Leakage checks performed:** (1) every feature is prior-window-only, engineered before any label logic
touches the target window; (2) the label-construction bug described in the code comment above, caught by
noticing an impossibly high ROC-AUC paired with a worse-than-random baseline — exactly the
"suspiciously perfect" signal to distrust rather than report.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(feat, groups=feat["client_hash_id"]))
train, test = feat.iloc[train_idx].copy(), feat.iloc[test_idx].copy()
print(f"train: {len(train):,} rows / {train['client_hash_id'].nunique()} clients")
print(f"test:  {len(test):,} rows / {test['client_hash_id'].nunique()} clients (held out entirely)")

def bucket_pos(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    if p <= 50: return "21-50"
    return "50+"
train["pos_bucket"] = train["avg_position_prior"].apply(bucket_pos)
test["pos_bucket"]  = test["avg_position_prior"].apply(bucket_pos)
expected_ctr = train.groupby("pos_bucket")["ctr_prior"].mean()
test["expected_ctr"] = test["pos_bucket"].map(expected_ctr)
test["ctr_gap"] = (test["expected_ctr"] - test["ctr_prior"]).clip(lower=0)
test["baseline_score"] = test["ctr_gap"] * test["impressions_prior"]

feature_cols_num = ["impressions_prior", "clicks_prior", "ctr_prior", "avg_position_prior",
                     "days_with_impr_prior", "content_age_days"]
train_X = pd.get_dummies(train[feature_cols_num + ["content_type"]], columns=["content_type"])
test_X = pd.get_dummies(test[feature_cols_num + ["content_type"]], columns=["content_type"])
test_X = test_X.reindex(columns=train_X.columns, fill_value=0)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(train_X, train["is_declining"])
test["model_score"] = rf.predict_proba(test_X)[:, 1]

base_rate_test = float(test["is_declining"].mean())
print(f"\nTest-set base rate (honest 'no-skill' floor - a random ranking scores this on average): {base_rate_test:.3f}")

rows = []
for name, scores in [("baseline (CTR-gap rule)", test["baseline_score"]), ("model (random forest)", test["model_score"])]:
    auc = roc_auc_score(test["is_declining"], scores)
    p20 = precision_at_k(scores, test["is_declining"], 20)
    p50 = precision_at_k(scores, test["is_declining"], 50)
    rows.append((name, round(auc,3), round(p20,3), round(p50,3)))
rows.append(("no-skill floor (test base rate)", 0.500, round(base_rate_test,3), round(base_rate_test,3)))
results_table = pd.DataFrame(rows, columns=["approach", "ROC-AUC", "Precision@20", "Precision@50"])
print("\n=== RESULTS (held-out clients) ===")
print(results_table.to_string(index=False))

# Grouped vs non-grouped: the memorization-gap check
Xall = pd.concat([train_X, test_X]); yall = pd.concat([train["is_declining"], test["is_declining"]])
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xall, yall, test_size=0.2, random_state=42, stratify=yall)
rf2 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1).fit(Xtr2, ytr2)
auc_nongrouped = roc_auc_score(yte2, rf2.predict_proba(Xte2)[:,1])
print(f"\nSame model, RANDOM row-level split (clients leak across train/test): ROC-AUC = {auc_nongrouped:.3f}")
print(f"Same model, CLIENT-GROUPED split (honest):                            ROC-AUC = {roc_auc_score(test['is_declining'], test['model_score']):.3f}")
print("The gap between these two is the memorization the grouped split is designed to catch.")


train: 107,030 rows / 31 clients
test:  7,685 rows / 8 clients (held out entirely)



Test-set base rate (honest 'no-skill' floor - a random ranking scores this on average): 0.509

=== RESULTS (held-out clients) ===
                       approach  ROC-AUC  Precision@20  Precision@50
        baseline (CTR-gap rule)    0.506         0.350         0.460
          model (random forest)    0.587         0.750         0.780
no-skill floor (test base rate)    0.500         0.509         0.509



Same model, RANDOM row-level split (clients leak across train/test): ROC-AUC = 0.684
Same model, CLIENT-GROUPED split (honest):                            ROC-AUC = 0.587
The gap between these two is the memorization the grouped split is designed to catch.


## 4. Results (vs baseline)

The model beats both the CTR-gap baseline and the honest no-skill floor (the test-set base rate — what a
random ranking scores on average) at Precision@20 and Precision@50, on clients it never saw during
training. The gap between the grouped and non-grouped ROC-AUC is reported explicitly rather than only
citing the flattering number, because that gap is exactly the kind of memorization a random split would
hide.

**Directional claim** (per the claim ladder: observed → directional, not causal): pages the model ranks
highest are, on average, more likely to show a meaningful demand drop in the next two weeks than pages
picked by the position-based CTR rule alone, or by no ranking at all. This is a statement about *relative
ranking quality on this sample*, not a guarantee about any individual page or client.


In [4]:
print("Limitations are written qualitatively below; the supporting numbers were computed in Section 4.")
print(f"- Test set: {test['client_hash_id'].nunique()} clients / {len(test):,} pages - a single month, one partition.")
print(f"- GSC coverage excluded ~63% of the raw partition before any modeling started (per the ML-04 contract).")
print(f"- Label prevalence (decline rate) in this sample: {feat['is_declining'].mean():.3f} - not necessarily stable across seasons or clients.")


Limitations are written qualitatively below; the supporting numbers were computed in Section 4.
- Test set: 8 clients / 7,685 pages - a single month, one partition.
- GSC coverage excluded ~63% of the raw partition before any modeling started (per the ML-04 contract).
- Label prevalence (decline rate) in this sample: 0.344 - not necessarily stable across seasons or clients.


## 5. Limitations

- **One month, one partition.** The label, baseline, and model are all built from March 2026 only. No
  claim here has been checked against a different month or season — a genuinely different time window
  could rank differently, especially if it includes a holiday, algorithm update, or reporting anomaly.
- **8 held-out clients.** The grouped test split has strong client-level honesty (no client appears in
  both train and test), but 8 clients is a small population to claim the model "generalizes to any new
  client." A single unusual client in that group of 8 can move the numbers.
- **GSC-only, ~37% of rows.** Roughly 63% of the raw partition was excluded up front because it had no
  GSC coverage (per the ML-04 data contract). The model has nothing to say about those pages; the queue
  is silent on them, not confident they're fine.
- **The label is a proxy, not a certified outcome.** "20% lower daily impression rate over 16 days" is a
  reasonable stand-in for "losing demand," but a page could dip for 16 days and recover, or could be
  flagged by a rate calculation on a small denominator. It measures a real signal, not ground truth.
- **This is a ranking aid, not an automated decision.** Precision@20/50 in the 0.7 range means roughly
  1 in 4-3 flagged pages is a false positive even at the model's best settings — acceptable for a
  reviewed queue, not for an unattended action.
- **Recommendation concentration (see Section 6).** A naive top-N pull by raw model score concentrates
  heavily on a small number of clients and very low-traffic pages — a real pattern in how the score
  distributes, not a bug, but one that would make a poor and non-representative showcase if presented
  without the filtering and diversification done below.


In [5]:
cols = ["content_hash_id","client_hash_id","content_type","impressions_prior","ctr_prior",
        "avg_position_prior","days_with_impr_prior","content_age_days","model_score"]

# --- What a naive "top 10 by model_score" pull looks like ---
raw_top10 = test.sort_values("model_score", ascending=False)[cols].head(10)
print(f"Naive top-10 by raw model_score: {raw_top10['client_hash_id'].nunique()} unique client(s), "
      f"dominant client share = {raw_top10['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}, "
      f"impressions_prior range = {raw_top10['impressions_prior'].min():.0f}-{raw_top10['impressions_prior'].max():.0f}")
print("This is barely a 'top 10 across the client base' - it is almost a top-10-within-one-client, and")
print("every impressions_prior value is far below the population median. Presenting this as-is would")
print("overstate how broadly useful the ranking is. See the filtered/diversified version below.")

# --- Filtered + diversified version actually used for the recommendations below ---
MIN_IMPR = 100  # excludes near-zero-volume noise; still well under the ~251 population median
qualified = test[test["impressions_prior"] >= MIN_IMPR].sort_values("model_score", ascending=False)

def diversified_topn(df, n=10, cap_per_client=2):
    picks, counts = [], {}
    for _, row in df.iterrows():
        c = row["client_hash_id"]
        if counts.get(c, 0) >= cap_per_client:
            continue
        picks.append(row); counts[c] = counts.get(c, 0) + 1
        if len(picks) == n:
            break
    return pd.DataFrame(picks)

recommendations = diversified_topn(qualified, n=10, cap_per_client=2)[cols].reset_index(drop=True)
recommendations["reason_code"] = "model_decline_risk_high_confidence"
recommendations["action"] = "editorial_review_refresh_candidate"
print(f"\nDiversified, volume-filtered top 10 (>= {MIN_IMPR} prior impressions, max 2 picks per client):")
print(f"{recommendations['client_hash_id'].nunique()} unique clients represented")
print(recommendations.to_string(index=False))

# --- Error examples: where the model is confidently wrong, so a reviewer knows what to double-check ---
fp = test[(test["model_score"]>0.6) & (test["is_declining"]==0) & (test["impressions_prior"]>=100)] \
        .sort_values("model_score", ascending=False)[cols].head(3)
fn = test[(test["model_score"]<0.3) & (test["is_declining"]==1) & (test["impressions_prior"]>=100)] \
        .sort_values("model_score")[cols].head(3)
print("\nConfidently-wrong FALSE POSITIVES (flagged high risk, was actually stable/growing):")
print(fp.to_string(index=False))
print("\nCONFIDENTLY-WRONG FALSE NEGATIVES (flagged low risk, actually declined):")
print(fn.to_string(index=False))
print("\nBoth error groups share a pattern worth a reviewer's attention: zero or near-zero ctr_prior and")
print("mid-range position. The model leans on content_age_days and ctr_prior most heavily (Section 7),")
print("so pages where those signals point in conflicting directions are exactly where it is least reliable.")


Naive top-10 by raw model_score: 2 unique client(s), dominant client share = 80%, impressions_prior range = 11-20
This is barely a 'top 10 across the client base' - it is almost a top-10-within-one-client, and
every impressions_prior value is far below the population median. Presenting this as-is would
overstate how broadly useful the ranking is. See the filtered/diversified version below.

Diversified, volume-filtered top 10 (>= 100 prior impressions, max 2 picks per client):
5 unique clients represented
         content_hash_id          client_hash_id    content_type  impressions_prior  ctr_prior  avg_position_prior  days_with_impr_prior  content_age_days  model_score                        reason_code                             action
content_c49689f071ca7e4d client_3ffa76342f366962  feedly article              138.0   0.000000            6.685917                     8               210     0.713833 model_decline_risk_high_confidence editorial_review_refresh_candidate
content_3fbb9

## 6. Ranked recommendations

**A naive top-10 by raw model score is a poor showcase and would overstate the ranking's reach:** it
concentrates on 1-2 clients and pages with impressions far below the sample median — the code below
prints this explicitly rather than hiding it.

**What the recommendations actually use instead:** the queue is filtered to pages with at least 100 prior
impressions (well under the median, but enough to exclude near-zero-volume noise) and capped at 2 picks
per client, so the top-10 below represents multiple clients and content types rather than one client's
long tail. This filtering is a presentation choice for *this section*, not a change to the model or the
Results numbers above, which are reported on the full, unfiltered test set.

**How to read each row:** `model_score` is a relative rank, not a probability a team should quote to a
client. `reason_code` and `action` are illustrative placeholders (mirroring the ML-05 baseline's
reason-code pattern) for what a production version of this queue would attach to each row for an editor.
Two confidently-wrong examples in each direction are included above so a reviewer knows what kind of page
to double-check by hand rather than trust the score blindly.


In [6]:
import matplotlib, json
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = f"{REPO_ROOT}/work/outputs"
os.makedirs(OUT, exist_ok=True)

# Chart 1: Results comparison (Precision@20/50 for baseline vs model vs floor)
fig, ax = plt.subplots(figsize=(6,4))
approaches = results_table["approach"].tolist()
x = np.arange(len(approaches)); width = 0.35
ax.bar(x - width/2, results_table["Precision@20"], width, label="Precision@20")
ax.bar(x + width/2, results_table["Precision@50"], width, label="Precision@50")
ax.set_xticks(x); ax.set_xticklabels(["baseline","model","no-skill\nfloor"], fontsize=9)
ax.set_ylabel("Precision"); ax.set_title("Model vs baseline vs no-skill floor (held-out clients)")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/capstone_results_comparison.png", dpi=120)
plt.close()

# Chart 2: feature importances
imp = pd.Series(rf.feature_importances_, index=train_X.columns).sort_values(ascending=True).tail(8)
fig, ax = plt.subplots(figsize=(6,4))
ax.barh(imp.index, imp.values, color="#3b6fa0")
ax.set_xlabel("Importance"); ax.set_title("Top feature importances (random forest)")
plt.tight_layout()
plt.savefig(f"{OUT}/capstone_feature_importance.png", dpi=120)
plt.close()

metrics_out = {
    "n_train": int(len(train)), "n_test": int(len(test)),
    "n_train_clients": int(train["client_hash_id"].nunique()),
    "n_test_clients": int(test["client_hash_id"].nunique()),
    "base_rate_train": round(float(train["is_declining"].mean()),3),
    "base_rate_test": round(base_rate_test,3),
    "results_table": results_table.to_dict(orient="records"),
    "model_roc_auc_nongrouped_comparison": round(float(auc_nongrouped),3),
    "top_features": {k: round(float(v),4) for k,v in imp.sort_values(ascending=False).items()},
    "raw_top10_unique_clients": int(raw_top10["client_hash_id"].nunique()),
    "diversified_top10_unique_clients": int(recommendations["client_hash_id"].nunique()),
    "min_impressions_filter": MIN_IMPR,
}
with open(f"{OUT}/capstone_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

print("Saved charts + metrics to work/outputs/:")
print(" - capstone_results_comparison.png")
print(" - capstone_feature_importance.png")
print(" - capstone_metrics.json")
print(json.dumps(metrics_out, indent=2))


Saved charts + metrics to work/outputs/:
 - capstone_results_comparison.png
 - capstone_feature_importance.png
 - capstone_metrics.json
{
  "n_train": 107030,
  "n_test": 7685,
  "n_train_clients": 31,
  "n_test_clients": 8,
  "base_rate_train": 0.333,
  "base_rate_test": 0.509,
  "results_table": [
    {
      "approach": "baseline (CTR-gap rule)",
      "ROC-AUC": 0.506,
      "Precision@20": 0.35,
      "Precision@50": 0.46
    },
    {
      "approach": "model (random forest)",
      "ROC-AUC": 0.587,
      "Precision@20": 0.75,
      "Precision@50": 0.78
    },
    {
      "approach": "no-skill floor (test base rate)",
      "ROC-AUC": 0.5,
      "Precision@20": 0.509,
      "Precision@50": 0.509
    }
  ],
  "model_roc_auc_nongrouped_comparison": 0.684,
  "top_features": {
    "content_age_days": 0.2951,
    "ctr_prior": 0.1686,
    "avg_position_prior": 0.1574,
    "days_with_impr_prior": 0.1406,
    "impressions_prior": 0.1252,
    "clicks_prior": 0.0818,
    "content_type_keyw

## 7. Artifacts the paper embeds

Two charts and one metrics file are generated below and saved to `work/outputs/`, to be embedded directly
in the deployed research paper: a Precision@K comparison (baseline vs model vs no-skill floor) and a
feature-importance chart. `capstone_metrics.json` holds every number quoted in this notebook and the
paper, so the paper's numbers can be traced back to a single generated file rather than retyped by hand.


In [7]:
print('artifacts generated in the cell above')

artifacts generated in the cell above


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only hashed IDs, counts, rates, dates
- [x] My claims use care — see Section 4's "directional claim" language and Section 5's limitations
- [x] My deployed paper has all 9 sections in order: Title+Abstract, Introduction/Problem, Data,
      Methodology, Results, Limitations, Ranked recommendations, Reproducibility, and Acknowledgments &
      data credit (linking flyrank.ai) at the bottom
- [x] ML-12 done in this notebook's closing cells: a 5-minute demo outline, a social-post cut, and a
      3-sentence employer-facing summary


## ML-12 — Communicating the work

**5-minute demo outline** (what I'd actually walk someone through, live):
1. *(30s)* The question: which of a client's pages are quietly losing search demand, early enough to act?
2. *(60s)* Show the data window split (prior 15 days -> target 16 days) and the label definition — a real
   20% drop in daily impression rate, and the label-leakage bug I caught and fixed before trusting any
   number (clicks-based label with a 0-click floor was mechanically rigging the result).
3. *(90s)* Show the Results table live: model vs CTR-gap baseline vs the honest no-skill floor, on
   clients the model never trained on, plus the grouped-vs-non-grouped ROC-AUC gap as the "why a random
   split would have lied to you" moment.
4. *(60s)* Show the naive top-10 (dominated by one client) next to the filtered, diversified top-10 —
   the honesty step of not shipping the flashier-looking but misleading version.
5. *(30s)* Limitations in one breath, then the ask: this ranks a review queue, it doesn't replace one.

**Social-post cut** (one paragraph, public-safe):
"Built a model to flag which content pages are about to lose search visibility — before it shows up in a
monthly report. The interesting part wasn't the model, it was catching my own label-leakage bug (a
0-click floor was quietly guaranteeing certain pages could never be labeled 'declining') and fixing it
before trusting the results. Final model beats a CTR-based heuristic and a random-ranking floor on
clients it never saw during training. Full writeup + honest limitations: [paper link]."

**3-sentence employer-facing summary:**
I framed a real content-operations question (which pages are losing search demand soon enough to act on
it) as a ranking problem, built prior-window-only features and a client-grouped validation split to avoid
memorization, and caught and fixed a genuine label-leakage bug in my own pipeline before reporting
results. The resulting model outperforms both a position-based heuristic baseline and the honest
no-skill floor at Precision@20/50 on held-out clients. I documented what the model can't claim (one
month of data, 8 held-out clients, ~63% of rows excluded for missing search-console coverage) as
carefully as what it can.
